# 🎓 AI Educational Pipeline - End to End
**PDF/Image → Text → Embeddings → LLM → Voice**

Pipeline:
1. marker → Text Extraction
2. Chunking
3. BGE-M3 → Embeddings + Vector Store (FAISS)
4. Whisper Turbo → Transcription
5. Persona Prompt
6. Gemma via Ollama → LLM Generation
7. XTTS v2 → Voice Output

## 📦 Step 0: Install Dependencies

In [ ]:
!pip install marker-pdf --force-reinstall
!pip install sentence-transformers
!pip install faiss-cpu
!pip install openai-whisper
!pip install ollama
!pip install flask pyngrok
!pip install numpy torch torchaudio

  Using cached marker_pdf-1.10.2-py3-none-any.whl.metadata (30 kB)
  Using cached pillow-10.4.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (9.2 kB)
  Using cached anthropic-0.46.0-py3-none-any.whl.metadata (23 kB)
  Using cached click-8.4.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
  Using cached ftfy-6.3.1-py3-none-any.whl.metadata (7.3 kB)
  Using cached google_genai-1.75.0-py3-none-any.whl.metadata (52 kB)
  Using cached markdown2-2.5.5-py3-none-any.whl.metadata (2.1 kB)
  Using cached markdownify-1.2.2-py3-none-any.whl.metadata (9.9 kB)
  Using cached openai-1.109.1-py3-none-any.whl.metadata (29 kB)
  Using cached pdftext-0.6.3-py3-none-any.whl.metadata (8.5 kB)
  Using cached pre_commit-4.6.0-py2.py3-none-any.whl.metadata (1.2 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached pydantic_settings-2.14.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached python_dotenv-1.2.2-py3-none-any

  Using cached numpy-2.0.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
Using cached numpy-2.0.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (19.2 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.6
    Uninstalling numpy-2.4.6:
      Successfully uninstalled numpy-2.4.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.25.0+cu128 requires torch==2.10.0, but you have torch 2.12.0 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.4.0 which is incompatible.
cudf-cu12 26.2.1 requires cuda-toolkit[nvcc,nvrtc]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.13.4 which is incompatible.
cuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand

  Using cached torch-2.10.0-3-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached cuda_bindings-12.9.4-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.6 kB)
  Using cached triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (1.7 kB)
Using cached torch-2.10.0-3-cp312-cp312-manylinux_2_28_x86_64.whl (915.6 MB)
Using cached cuda_bindings-12.9.4-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (12.2 MB)
Using cached triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (188.3 MB)
  Attempting uninstall: triton
    Found existing installation: triton 3.7.0
    Uninstalling triton-3.7.0:
      Successfully uninstalled triton-3.7.0
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
  Attempting uninstall: torch
    Found existing installation: torch 2.12.0
    Un

## 📄 Step 1: Text Extraction with Marker

In [ ]:
!pip install marker-pdf

In [ ]:
!apt-get install -y libmagic1 tesseract-ocr poppler-utils

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
libmagic1 is already the newest version (1:5.41-3ubuntu0.1).
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.


In [ ]:
# الإصدار الجديد من marker
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.config.parser import ConfigParser
import os

# Load marker models (once)
config = ConfigParser({"langs": ["Arabic", "English"]})
marker_models = create_model_dict()

def extract_text_marker(file_path: str) -> str:
    converter = PdfConverter(
        config=config.generate_config_dict(),
        artifact_dict=marker_models,
        processor_list=config.get_processors(),
        renderer=config.get_renderer()
    )
    rendered = converter(file_path)
    full_text = rendered.markdown
    print(f"✅ Extracted {len(full_text)} characters from {os.path.basename(file_path)}")
    return full_text

In [ ]:
try:
    from marker.convert import convert_single_pdf
    print("✅ `marker.convert` imported successfully!")
except ModuleNotFoundError:
    print("❌ `marker.convert` still not found. Please ensure the runtime was restarted and all cells, including the pip install cell, were run again.")

## ✂️ Step 2: Chunking

In [ ]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    """
    Split text into overlapping chunks.
    chunk_size: number of characters per chunk
    overlap: overlap between chunks to preserve context
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if chunk.strip():
            chunks.append(chunk.strip())
        start += chunk_size - overlap
    print(f"✅ Created {len(chunks)} chunks")
    return chunks

# ---- TEST ----
# chunks = chunk_text(extracted_text)
# print(chunks[0])

## 🔢 Step 3: BGE-M3 Embeddings + FAISS Vector Store

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load BGE-M3
print("Loading BGE-M3...")
embedder = SentenceTransformer("BAAI/bge-m3")
print("✅ BGE-M3 loaded")

# Global store
vector_store = {"index": None, "chunks": []}

def build_vector_store(chunks: list[str]):
    """Embed chunks and build FAISS index."""
    print("Embedding chunks...")
    embeddings = embedder.encode(chunks, show_progress_bar=True, normalize_embeddings=True)
    embeddings = np.array(embeddings).astype("float32")

    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # Inner product (cosine with normalized vectors)
    index.add(embeddings)

    vector_store["index"] = index
    vector_store["chunks"] = chunks
    print(f"✅ Vector store built with {index.ntotal} vectors")

def search_vector_store(query: str, top_k: int = 3) -> list[str]:
    """Search for most relevant chunks."""
    query_emb = embedder.encode([query], normalize_embeddings=True)
    query_emb = np.array(query_emb).astype("float32")

    scores, indices = vector_store["index"].search(query_emb, top_k)
    results = [vector_store["chunks"][i] for i in indices[0]]
    return results

# ---- TEST ----
# build_vector_store(chunks)
# results = search_vector_store("ما هو الموضوع الرئيسي؟")
# print(results)

## 🎙️ Step 4: Whisper Turbo - Voice Transcription

In [ ]:
import whisper

print("Loading Whisper Turbo...")
whisper_model = whisper.load_model("turbo")
print("✅ Whisper Turbo loaded")

def transcribe_audio(audio_path: str) -> str:
    """
    Transcribe voice question to text.
    Supports Arabic and other languages.
    """
    result = whisper_model.transcribe(
        audio_path,
        language="ar",  # Arabic
        task="transcribe"
    )
    text = result["text"].strip()
    print(f"✅ Transcribed: {text}")
    return text

# ---- TEST ----
# question_text = transcribe_audio("/content/question.wav")
# print(question_text)

## 📝 Step 5: Persona Prompt Builder

In [ ]:
def build_persona_prompt(question: str, context_chunks: list[str]) -> str:
    """
    Build prompt with context for Gemma.
    Uses Syrian Arabic dialect.
    """
    context = "\n\n".join(context_chunks)

    prompt = f"""أنت مساعد تعليمي ذكي. أجب على سؤال الطالب باللهجة السورية بشكل واضح ومبسط.
استخدم المعلومات التالية للإجابة:

=== المحتوى التعليمي ===
{context}

=== سؤال الطالب ===
{question}

=== الجواب (باللهجة السورية) ==="""

    return prompt

# ---- TEST ----
# prompt = build_persona_prompt(question_text, results)
# print(prompt)

## 🤖 Step 6: Gemma via Ollama - LLM Generation

In [ ]:
# Install zstd first, then Ollama
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve &
import time; time.sleep(5)
!ollama pull gemma3

In [ ]:
import ollama

def generate_answer(prompt: str, model: str = "gemma3") -> str:
    """
    Generate answer using Gemma via Ollama.
    """
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    answer = response["message"]["content"].strip()
    print(f"✅ Generated answer: {answer[:100]}...")
    return answer

# ---- TEST ----
# answer = generate_answer(prompt)
# print(answer)

## 🔊 Step 7: XTTS v2 - Voice Output

In [ ]:
!pip install condacolab
import condacolab
condacolab.install()

In [ ]:
!pip install TTS

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

try:
    from TTS.api import TTS
    print("Loading XTTS v2...")
    tts_model = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
    print("✅ XTTS v2 loaded")

    def text_to_speech(text: str, speaker_wav: str, output_path: str = "/content/output.wav") -> str:
        """
        Convert text to speech using cloned voice.
        speaker_wav: path to reference voice sample (3-10 seconds)
        """
        tts_model.tts_to_file(
            text=text,
            speaker_wav=speaker_wav,
            language="ar",
            file_path=output_path
        )
        print(f"✅ Audio saved to {output_path}")
        return output_path

    # ---- TEST ----
    # audio_path = text_to_speech(answer, "/content/speaker_sample.wav")
    # from IPython.display import Audio
    # Audio(audio_path)

except ModuleNotFoundError:
    print("❌ `TTS` module not found. This might be due to environment changes (e.g., condacolab) not being fully applied.")
    print("Please ensure you have restarted the Colab runtime and re-run all installation cells.")
    # Define a dummy text_to_speech function to prevent further errors if execution continues
    def text_to_speech(text: str, speaker_wav: str, output_path: str = "/content/output.wav") -> str:
        print("Text-to-speech functionality is unavailable due to missing TTS module.")
        return ""

## 🚀 Step 8: Full Pipeline - End to End

In [ ]:
from IPython.display import Audio, display

def run_pipeline(
    document_path: str,      # PDF or image
    question_audio_path: str, # voice question
    speaker_wav_path: str,   # reference voice for cloning
    output_audio_path: str = "/content/answer.wav"
) -> str:
    """
    Full pipeline:
    Document + Voice Question → Voice Answer (cloned)
    """
    print("="*50)
    print("🚀 Starting AI Pipeline...")
    print("="*50)

    # Step 1: Extract text
    print("\n📄 Step 1: Extracting text...")
    text = extract_text_marker(document_path)

    # Step 2: Chunk
    print("\n✂️ Step 2: Chunking text...")
    chunks = chunk_text(text)

    # Step 3: Build vector store
    print("\n🔢 Step 3: Building vector store...")
    build_vector_store(chunks)

    # Step 4: Transcribe question
    print("\n🎙️ Step 4: Transcribing question...")
    question = transcribe_audio(question_audio_path)

    # Step 5: Search relevant context
    print("\n🔍 Step 5: Searching context...")
    context = search_vector_store(question, top_k=3)

    # Step 6: Build prompt & generate answer
    print("\n🤖 Step 6: Generating answer with Gemma...")
    prompt = build_persona_prompt(question, context)
    answer = generate_answer(prompt)

    # Step 7: Convert to speech
    print("\n🔊 Step 7: Converting to voice...")
    audio_path = text_to_speech(answer, speaker_wav_path, output_audio_path)

    print("\n" + "="*50)
    print("✅ Pipeline complete!")
    print(f"📝 Question: {question}")
    print(f"💬 Answer: {answer[:200]}...")
    print("="*50)

    # Play audio in Colab
    display(Audio(audio_path))
    return audio_path


# ==============================
# RUN THE PIPELINE
# ==============================
# Upload your files first then:

# run_pipeline(
#     document_path="/content/textbook.pdf",
#     question_audio_path="/content/question.wav",
#     speaker_wav_path="/content/speaker.wav"
# )

## 🌐 Step 9: Flask API (Optional - for Frontend Integration)
شغّلي هاد الـ cell إذا بدك توصلي الـ pipeline بالـ frontend

In [ ]:
from flask import Flask, request, jsonify, send_file
from pyngrok import ngrok
import threading
import os

app = Flask(__name__)

@app.route("/pipeline", methods=["POST"])
def pipeline_endpoint():
    """
    POST /pipeline
    Form data:
      - document: PDF or image file
      - question: audio file (wav)
      - speaker: reference voice file (wav)
    Returns: audio/wav
    """
    try:
        doc = request.files["document"]
        question_audio = request.files["question"]
        speaker_audio = request.files["speaker"]

        # Save uploaded files
        doc_path = f"/tmp/{doc.filename}"
        q_path = "/tmp/question.wav"
        spk_path = "/tmp/speaker.wav"
        out_path = "/tmp/answer.wav"

        doc.save(doc_path)
        question_audio.save(q_path)
        speaker_audio.save(spk_path)

        # Run pipeline
        run_pipeline(doc_path, q_path, spk_path, out_path)

        return send_file(out_path, mimetype="audio/wav")

    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok"})

# Start Flask in background thread
def run_flask():
    app.run(port=5000, debug=False, use_reloader=False)

thread = threading.Thread(target=run_flask)
thread.daemon = True
thread.start()

# Expose via ngrok
NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN" # <-- Replace with your actual ngrok token

if NGROK_AUTH_TOKEN == "YOUR_NGROK_TOKEN":
    print("❌ ngrok authentication token is missing or incorrect.")
    print("Please get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken")
    print("and replace 'YOUR_NGROK_TOKEN' in the code.")
else:
    try:
        ngrok.set_auth_token(NGROK_AUTH_TOKEN)
        public_url = ngrok.connect(5000)
        print(f"\n🌐 Public URL: {public_url}")
        print(f"📡 Pipeline endpoint: {public_url}/pipeline")
    except Exception as e:
        print(f"❌ Failed to connect ngrok: {e}")
        print("Please double-check your ngrok authentication token.")